# Mini-projet : chats ou chiens ?

**Semaine 5 · Jour 5** — Classification d'images binaire avec un CNN et de l'augmentation de données  
`Obligatoire` `GPU` `Keras`

**Objectifs**

- Prétraiter des images pour un CNN
- Appliquer l'augmentation de données pour mieux généraliser
- Construire et entraîner un CNN de classification binaire
- Utiliser le dropout contre le surapprentissage

> **Comment travailler avec ce notebook**
>
> 1. **Fichier → Enregistrer une copie dans Drive** : tu travailles sur ta propre copie.
> 2. Exécute les cellules de code fournies (données, code de départ), puis écris tes réponses dans les cellules **✍️ Ton code**.
> 3. Si un exercice utilise un jeu de données à télécharger (Kaggle, lien…), charge-le dans Colab (icône 📁 à gauche, puis « Importer »).
> 4. Pour rendre l'exercice : **Fichier → Enregistrer une copie dans GitHub**, dans ton dépôt `DI_Bootcamp`, chemin `Week5/Day5/MiniProjetClassificationChatsChiens/mini-projet-classification-chats-chiens.ipynb`.
> 5. Sur Propilote-Learning, colle le lien GitHub du dossier ou du fichier dans l'exercice correspondant.

## I. Mise en place

> ⚠️ **Important** : Ce projet demande beaucoup de calcul : utilise un GPU (Colab) ou une machine virtuelle (par exemple DigitalOcean).

- Télécharge le jeu de données (lien dans le cours du jour), décompresse-le, renomme le dossier en cats_dogs et place-le dans un dossier data.
- Les images sont en très haute résolution : réduis-la via IMG_HEIGHT et IMG_WIDTH (par exemple 48 × 48 ou 180 × 180).
- Certaines parties sont fournies (à copier telles quelles), les autres sont marquées « À faire ». Suis les étapes dans l'ordre.

In [ ]:
# ✍️ Ton code — exercice I : Mise en place


## II. Chargement des données (fourni)

Ce bloc trouve les images de cats_dogs/train/train et cats_dogs/test/test, déduit l'étiquette du dossier parent ou du nom de fichier (cat.123.jpg), puis crée trois générateurs : train_flow (avec augmentation), val_flow (mise à l'échelle seulement) et test_flow (non étiqueté, pour les prédictions).

In [ ]:
import os, re
from glob import glob
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
np.random.seed(42); tf.random.set_seed(42)

DATA_ROOT = Path("data/cats_dogs")
train_dir = DATA_ROOT / "train" / "train"
test_dir = DATA_ROOT / "test" / "test"
IMG_HEIGHT, IMG_WIDTH = 180, 180
batch_size, seed = 32, 1337

def build_df_from_folder(folder: Path, labeled: bool = True):
    files = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
        files.extend(glob(str(folder / "**" / ext), recursive=True))
    if not files:
        raise FileNotFoundError(f"No images found under {folder}")
    rows = []
    for f in files:
        if not labeled:
            rows.append({"filepath": f}); continue
        name, parent = Path(f).name.lower(), Path(f).parent.name.lower()
        if parent in {"cat", "cats"} or re.search(r"(^|[^a-z])cat([^a-z]|$)", name):
            rows.append({"filepath": f, "label": "cat"})
        elif parent in {"dog", "dogs"} or re.search(r"(^|[^a-z])dog([^a-z]|$)", name):
            rows.append({"filepath": f, "label": "dog"})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full = build_df_from_folder(test_dir, labeled=False)
df_tr, df_val = train_test_split(df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=seed)

train_gen = ImageDataGenerator(rescale=1./255, rotation_range=45, width_shift_range=0.15,
                               height_shift_range=0.15, zoom_range=0.5, horizontal_flip=True)
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

flow_args = dict(x_col="filepath", target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=batch_size, validate_filenames=False)
train_flow = train_gen.flow_from_dataframe(df_tr, y_col="label", class_mode="binary", shuffle=True, seed=seed, **flow_args)
val_flow = val_gen.flow_from_dataframe(df_val, y_col="label", class_mode="binary", shuffle=False, **flow_args)
test_flow = test_gen.flow_from_dataframe(df_test_full, y_col=None, class_mode=None, shuffle=False, **flow_args)

> 💡 **Pourquoi ?** : On standardise la taille et l'échelle des pixels pour stabiliser les gradients ; on n'augmente que l'entraînement ; la validation donne un signal non biaisé pour guider les choix.

In [ ]:
# ✍️ Ton code — exercice II : Chargement des données (fourni)


## III. Les étapes à réaliser

### 2. Inspecter les données

- Décris en un paragraphe le nombre d'images par classe (train_flow.class_indices, train_flow.labels), l'équilibre des classes et les sources de variabilité (pose, échelle, éclairage, arrière-plan).
- Affiche une grille d'images d'entraînement avec leur étiquette, et explique quels indices visuels peuvent aider le modèle.

### 3. Définir l'architecture

Décris ton CNN en phrases avant de le coder : nombre de blocs convolutifs et taille des filtres, placement du MaxPooling, usage du Dropout, couches denses finales avec une sortie sigmoïde unique.

> 💡 **À retenir** : Entropie croisée binaire avec une sortie sigmoïde pour deux classes ; softmax est réservé au multiclasse.

### 4. Choisir l'optimisation

Précise et justifie : l'optimiseur (Adam recommandé), le taux d'apprentissage initial, la taille de lot selon la mémoire, EarlyStopping sur la perte de validation et, en option, ReduceLROnPlateau.

### 5. Entraîner

Entraîne sur un nombre fixe d'époques en traçant les courbes, puis recommence avec l'arrêt précoce. Explique comment tu détectes le surapprentissage et ce que tu changes pour le corriger.

### 6. Évaluer sur la validation

Rapporte l'exactitude et la perte, une matrice de confusion, la précision et le rappel par classe. Quel type d'erreur domine, et qu'est-ce que cela implique (seuil, augmentation) ? Le seuil de 0,5 est arbitraire : ajuste-le si besoin.

### 7. Prédire sur le test

Calcule les probabilités sur test_flow, convertis-les en étiquettes avec un seuil justifié et enregistre un CSV avec les colonnes filepath, prob_dog et pred_label. Explique comment tu vérifierais manuellement un échantillon.

### 8. Référence sans augmentation

Entraîne le même modèle sans augmentation, compare métriques et courbes, et analyse l'écart de généralisation.

### 9. Déséquilibre des classes

Si les classes sont déséquilibrées, calcule des poids de classe, réentraîne avec class_weight et explique l'effet sur la précision et le rappel de la classe minoritaire.

### 10. Sauvegarder

Enregistre le meilleur modèle (H5 ou SavedModel) et la configuration d'entraînement (petit YAML ou JSON). Pourquoi faut-il sauvegarder poids ET métadonnées ?

### 11. Extension

Propose et justifie une amélioration : Batch Normalization après les convolutions, transfer learning avec un MobileNetV2 gelé et une petite tête de classification, ou entraînement en précision mixte.

In [ ]:
# ✍️ Ton code — exercice III : Les étapes à réaliser


## IV. Livrables

- un court rapport sur les données (effectifs par classe, grille d'exemples) ;
- la description du modèle et les choix d'optimisation, rédigés ;
- les courbes d'entraînement et de validation, interprétées ;
- les métriques de validation (matrice de confusion, précision, rappel) ;
- le CSV des prédictions sur le test ;
- le modèle sauvegardé et un bref journal d'exécution.

Pousse ton code, ton modèle et tes graphiques sur GitHub, puis soumets le lien du dépôt (ou du notebook public) sur la plateforme.

In [ ]:
# ✍️ Ton code — exercice IV : Livrables


---

✅ **Terminé ?** Vérifie que toutes tes cellules s'exécutent sans erreur (**Exécution → Tout exécuter**), puis enregistre une copie dans GitHub et rends le lien sur Propilote-Learning.